# Stage 1 — Run classifiers on the Ethiopia moderation dataset

**Purpose.** Produce predictions from three classifiers on the audit-eligible
rows of the cleaned dataset, then save them to `predictions.xlsx` for Stage 2
(evaluation).

**Classifiers:**

1. **Perspective API (Google)** — generic toxicity baseline. Requires API key.
2. **AfriHate (AfroXLMR)** — Africa-centric hate-speech classifier. Hugging Face.
3. **Amharic mBERT** — Amharic-only fine-tuned mBERT. Hugging Face. Runs on Amharic rows only.

**Setup.** Colab (Runtime → Change runtime type → T4 GPU).
Upload the cleaned dataset xlsx when prompted.

**Runtime estimate.** ~15–30 minutes on a T4 GPU depending on Perspective API rate limits.


In [ ]:
!pip install -q google-api-python-client

## 0. Install dependencies

## 1. Imports and setup

## 2. Upload the cleaned dataset

Upload `ethiopia_moderation_dataset_1000_cleaned.xlsx` when prompted.

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from google.colab import files

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


In [ ]:
df_all = pd.read_excel(DATASET_FILE, sheet_name='Dataset')
df = df_all[df_all['IncludeInAudit'] == True].copy().reset_index(drop=True)
print(f'Audit-eligible rows: {len(df)}')
print(df[['PostID', 'Language', 'Label3Class']].head())


NameError: name 'DATASET_FILE' is not defined

## 3. Classifier 1 — Perspective API

**Before running.** Get an API key at https://developers.perspectiveapi.com. Paste it
into the cell below. Perspective API is rate-limited (1 QPS on free tier); the
cell below spaces requests accordingly.

Perspective returns continuous `TOXICITY` scores in `[0, 1]`. We convert to a
binary prediction using the standard 0.5 threshold (methodological note §4).


In [ ]:
print("Step 1: Starting")
try:
    from googleapiclient import discovery
    print("Step 2: Imported discovery module")

    client = discovery.build(
        'commentanalyzer', 'v1alpha1',
        developerKey=PERSPECTIVE_API_KEY,
        discoveryServiceUrl='https://commentanalyzer.googleapis.com/$discovery/rest?version=v1alpha1',
        static_discovery=False,
    )
    print("Step 3: Client built")

    test_text = df.iloc[0]['OriginalText']
    print(f"Step 4: Test text: {test_text[:60]}")

    req = {
        'comment': {'text': test_text},
        'requestedAttributes': {'TOXICITY': {}},
        'doNotStore': True,
    }
    print("Step 5: Calling API now...")

    resp = client.comments().analyze(body=req).execute()
    print(f"Step 6: Got response: {resp}")

except Exception as e:
    print(f"ERROR at current step: {type(e).__name__}")
    print(f"Message: {e}")
    import traceback
    traceback.print_exc()

print("End of diagnostic")

Step 1: Starting
Step 2: Imported discovery module
ERROR at current step: NameError
Message: name 'PERSPECTIVE_API_KEY' is not defined
End of diagnostic


Traceback (most recent call last):
  File "/tmp/ipykernel_14742/2391567368.py", line 8, in <cell line: 0>
    developerKey=PERSPECTIVE_API_KEY,
                 ^^^^^^^^^^^^^^^^^^^
NameError: name 'PERSPECTIVE_API_KEY' is not defined


In [ ]:
PERSPECTIVE_API_KEY = ''
if not PERSPECTIVE_API_KEY:
    print('Skipping Perspective API (no key provided).')
    perspective_scores = [None] * len(df)
else:
    from googleapiclient import discovery

    client = discovery.build(
        'commentanalyzer', 'v1alpha1',
        developerKey=PERSPECTIVE_API_KEY,
        discoveryServiceUrl='https://commentanalyzer.googleapis.com/$discovery/rest?version=v1alpha1',
        static_discovery=False,
    )

    def score_toxicity(text, lang_code):
        # Perspective supports Amharic (am). Afan Oromo is not officially supported;
        # we pass languages=[] to let Perspective auto-detect for Oromo rows.
        languages = [lang_code] if lang_code else []
        req = {
            'comment': {'text': text},
            'requestedAttributes': {'TOXICITY': {}},
            'languages': languages,
            'doNotStore': True,
        }
        try:
            resp = client.comments().analyze(body=req).execute()
            return resp['attributeScores']['TOXICITY']['summaryScore']['value']
        except Exception as e:
            return None

    lang_map = {'Amharic': 'am', 'Afan Oromo': ''}
    perspective_scores = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Perspective'):
        score = score_toxicity(row['OriginalText'], lang_map.get(row['Language'], ''))
        perspective_scores.append(score)
        time.sleep(1.05)  # stay under 1 QPS

df['perspective_score'] = perspective_scores
df['perspective_pred_hate'] = [
    (s is not None and s >= 0.5) for s in perspective_scores
]
df[['PostID', 'Language', 'perspective_score', 'perspective_pred_hate']].head()


Perspective:   0%|          | 0/789 [00:00<?, ?it/s]

,PostID,Language,perspective_score,perspective_pred_hate
0,POST_0001,Amharic,None,False
1,POST_0002,Afan Oromo,None,False
2,POST_0003,Amharic,None,False
3,POST_0004,Afan Oromo,None,False
4,POST_0005,Afan Oromo,None,False


In [ ]:
PERSPECTIVE_API_KEY = 'YOUR_PERSPECTIVE_API_KEY'

import time
import traceback
from googleapiclient import discovery

client = discovery.build(
    'commentanalyzer', 'v1alpha1',
    developerKey=PERSPECTIVE_API_KEY,
    discoveryServiceUrl='https://commentanalyzer.googleapis.com/$discovery/rest?version=v1alpha1',
    static_discovery=False,
)

def score_forced_en(text):
    """Force Perspective to treat text as English. Returns (score, status)."""
    req = {
        'comment': {'text': text},
        'requestedAttributes': {'TOXICITY': {}},
        'languages': ['en'],   # FORCED — simulates naive deployment
        'doNotStore': True,
    }
    try:
        resp = client.comments().analyze(body=req).execute()
        return resp['attributeScores']['TOXICITY']['summaryScore']['value'], 'ok'
    except Exception as e:
        return None, type(e).__name__

scores, statuses = [], []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Perspective (forced EN)'):
    s, st = score_forced_en(row['OriginalText'])
    scores.append(s)
    statuses.append(st)
    time.sleep(1.05)

df['perspective_forced_en_score'] = scores
df['perspective_forced_en_status'] = statuses
df['perspective_forced_en_pred_hate'] = [
    (s is not None and s >= 0.5) for s in scores
]

# Also record the "native" coverage finding for the file
df['perspective_native_unsupported'] = True   # universal for both languages — confirmed via diagnostic

import pandas as pd
print(f"Forced-EN statuses: {pd.Series(statuses).value_counts().to_dict()}")
print(f"Forced-EN scored successfully: {sum(1 for st in statuses if st == 'ok')} / {len(df)}")
df[['PostID', 'Language', 'perspective_forced_en_score',
    'perspective_forced_en_pred_hate', 'perspective_forced_en_status']].head()

HttpError: <HttpError 400 when requesting https://commentanalyzer.googleapis.com/$discovery/rest?version=v1alpha1&key=YOUR_PERSPECTIVE_API_KEY returned "API Key not found. Please pass a valid API key.". Details: "[{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'commentanalyzer.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API Key not found. Please pass a valid API key.'}]">

In [ ]:
try:
    print(f"Key is set: length {len(PERSPECTIVE_API_KEY)}, starts with {PERSPECTIVE_API_KEY[:6]}")
except NameError:
    print("Runtime lost state — need to re-run Cell 2 and re-paste key in Cell 5's first line")

Key is set: length 39, starts with AIzaSy


In [ ]:
import traceback
from googleapiclient import discovery

print("Step 1: Building client...")
try:
    client = discovery.build(
        'commentanalyzer', 'v1alpha1',
        developerKey=PERSPECTIVE_API_KEY,
        discoveryServiceUrl='https://commentanalyzer.googleapis.com/$discovery/rest?version=v1alpha1',
        static_discovery=False,
    )
    print("Step 1: OK — client built")
except Exception as e:
    print(f"Step 1 FAILED: {type(e).__name__}: {e}")
    traceback.print_exc()
    raise

print("\nStep 2: Making test API call with English text...")
test_text = "Hello world, this is a test."
req = {
    'comment': {'text': test_text},
    'requestedAttributes': {'TOXICITY': {}},
    'doNotStore': True,
}

try:
    resp = client.comments().analyze(body=req).execute()
    print("Step 2: OK — response received:")
    print(resp)
except Exception as e:
    print(f"Step 2 FAILED: {type(e).__name__}")
    print(f"Error: {e}")
    if hasattr(e, 'content'):
        print(f"\nHTTP error content:\n{e.content.decode() if isinstance(e.content, bytes) else e.content}")
    if hasattr(e, 'resp'):
        print(f"\nHTTP status code: {e.resp.status}")
    traceback.print_exc()

Step 1: Building client...
Step 1: OK — client built

Step 2: Making test API call with English text...
Step 2: OK — response received:
{'attributeScores': {'TOXICITY': {'spanScores': [{'begin': 0, 'end': 28, 'score': {'value': 0.018974753, 'type': 'PROBABILITY'}}], 'summaryScore': {'value': 0.018974753, 'type': 'PROBABILITY'}}}, 'languages': ['en'], 'detectedLanguages': ['en']}


In [ ]:
import traceback

amh_row = df[df['Language'] == 'Amharic'].iloc[0]
print(f"Testing PostID: {amh_row['PostID']}")
print(f"Text: {amh_row['OriginalText']}")
print()

# Try WITH explicit 'am' language
print("=== Attempt A: languages=['am'] ===")
try:
    resp = client.comments().analyze(body={
        'comment': {'text': amh_row['OriginalText']},
        'requestedAttributes': {'TOXICITY': {}},
        'languages': ['am'],
        'doNotStore': True,
    }).execute()
    print("SUCCESS:", resp)
except Exception as e:
    print(f"FAILED ({type(e).__name__}): {e}")
    if hasattr(e, 'content'):
        print(f"Content: {e.content.decode() if isinstance(e.content, bytes) else e.content}")

print("\n=== Attempt B: no language (auto-detect) ===")
try:
    resp = client.comments().analyze(body={
        'comment': {'text': amh_row['OriginalText']},
        'requestedAttributes': {'TOXICITY': {}},
        'doNotStore': True,
    }).execute()
    print("SUCCESS:", resp)
except Exception as e:
    print(f"FAILED ({type(e).__name__}): {e}")
    if hasattr(e, 'content'):
        print(f"Content: {e.content.decode() if isinstance(e.content, bytes) else e.content}")

Testing PostID: POST_0001
Text: እነዚህ ሰዎች አገራችንን እያፈረሱ ነው

=== Attempt A: languages=['am'] ===
FAILED (HttpError): <HttpError 400 when requesting https://commentanalyzer.googleapis.com/v1alpha1/comments:analyze?key=YOUR_PERSPECTIVE_API_KEY&alt=json returned "Attribute TOXICITY does not support request languages: am". Details: "[{'@type': 'type.googleapis.com/google.commentanalyzer.v1alpha1.Error', 'errorType': 'LANGUAGE_NOT_SUPPORTED_BY_ATTRIBUTE', 'languageNotSupportedByAttributeError': {'requestedLanguages': ['am'], 'attribute': 'TOXICITY'}}]">
Content: {
  "error": {
    "code": 400,
    "message": "Attribute TOXICITY does not support request languages: am",
    "status": "INVALID_ARGUMENT",
    "details": [
      {
        "@type": "type.googleapis.com/google.commentanalyzer.v1alpha1.Error",
        "errorType": "LANGUAGE_NOT_SUPPORTED_BY_ATTRIBUTE",
        "languageNotSupportedByAttributeError": {
          "requestedLanguages": [
            "am"
          ],
          "attribute

In [ ]:
oro_row = df[df['Language'] == 'Afan Oromo'].iloc[0]
print(f"Testing PostID: {oro_row['PostID']}")
print(f"Text: {oro_row['OriginalText']}")
print()

print("=== No language (auto-detect) ===")
try:
    resp = client.comments().analyze(body={
        'comment': {'text': oro_row['OriginalText']},
        'requestedAttributes': {'TOXICITY': {}},
        'doNotStore': True,
    }).execute()
    print("SUCCESS:", resp)
except Exception as e:
    if hasattr(e, 'content'):
        import json
        content = e.content.decode() if isinstance(e.content, bytes) else e.content
        print("FAILED. Error body:")
        print(content)
    else:
        print(f"FAILED: {e}")

Testing PostID: POST_0002
Text: Waliin dhaabbannee cunqursaa dura dhaabbachuu qabna

=== No language (auto-detect) ===
FAILED. Error body:
{
  "error": {
    "code": 400,
    "message": "Attribute TOXICITY does not support request languages: om",
    "status": "INVALID_ARGUMENT",
    "details": [
      {
        "@type": "type.googleapis.com/google.commentanalyzer.v1alpha1.Error",
        "errorType": "LANGUAGE_NOT_SUPPORTED_BY_ATTRIBUTE",
        "languageNotSupportedByAttributeError": {
          "detectedLanguages": [
            "om"
          ],
          "attribute": "TOXICITY"
        }
      }
    ]
  }
}



In [ ]:
PERSPECTIVE_API_KEY = 'YOUR_PERSPECTIVE_API_KEY'

import time
import pandas as pd
from googleapiclient import discovery

client = discovery.build(
    'commentanalyzer', 'v1alpha1',
    developerKey=PERSPECTIVE_API_KEY,
    discoveryServiceUrl='https://commentanalyzer.googleapis.com/$discovery/rest?version=v1alpha1',
    static_discovery=False,
)

def score_forced_en(text):
    req = {
        'comment': {'text': text},
        'requestedAttributes': {'TOXICITY': {}},
        'languages': ['en'],
        'doNotStore': True,
    }
    try:
        resp = client.comments().analyze(body=req).execute()
        return resp['attributeScores']['TOXICITY']['summaryScore']['value'], 'ok'
    except Exception as e:
        return None, type(e).__name__

scores, statuses = [], []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Perspective (forced EN)'):
    s, st = score_forced_en(row['OriginalText'])
    scores.append(s)
    statuses.append(st)
    time.sleep(1.05)

df['perspective_forced_en_score'] = scores
df['perspective_forced_en_status'] = statuses
df['perspective_forced_en_pred_hate'] = [
    (s is not None and s >= 0.5) for s in scores
]
df['perspective_native_unsupported'] = True

print(f"Forced-EN statuses: {pd.Series(statuses).value_counts().to_dict()}")
print(f"Forced-EN scored successfully: {sum(1 for st in statuses if st == 'ok')} / {len(df)}")
df[['PostID', 'Language', 'perspective_forced_en_score',
    'perspective_forced_en_pred_hate', 'perspective_forced_en_status']].head()

Perspective (forced EN):   0%|          | 0/789 [00:00<?, ?it/s]

Forced-EN statuses: {'ok': 789}
Forced-EN scored successfully: 789 / 789


,PostID,Language,perspective_forced_en_score,perspective_forced_en_pred_hate,perspective_forced_en_status
0,POST_0001,Amharic,0.239619,False,ok
1,POST_0002,Afan Oromo,0.087165,False,ok
2,POST_0003,Amharic,0.165053,False,ok
3,POST_0004,Afan Oromo,0.024614,False,ok
4,POST_0005,Afan Oromo,0.057748,False,ok


In [ ]:
OUT = 'predictions_v2.xlsx'

available_cols = ['PostID']
for col in ['mbert_pred_hate',
            'perspective_forced_en_score',
            'perspective_forced_en_pred_hate',
            'perspective_forced_en_status',
            'perspective_native_unsupported']:
    if col in df.columns:
        available_cols.append(col)

df[available_cols].to_excel(OUT, index=False)
print(f'Saved: {OUT} with {len(df)} rows')
print(f'Columns saved: {available_cols}')

if 'mbert_pred_hate' in df.columns:
    print(f"mBERT predictions present: {df['mbert_pred_hate'].notna().sum()} rows")
else:
    print('WARNING: mBERT predictions are NOT in df — you will need to rerun the mBERT cells before saving.')

if 'perspective_forced_en_pred_hate' in df.columns:
    print(f"Forced-EN predictions present: {df['perspective_forced_en_pred_hate'].notna().sum()} rows")

from google.colab import files
files.download(OUT)

Saved: predictions_v2.xlsx with 789 rows
Columns saved: ['PostID', 'mbert_pred_hate', 'perspective_forced_en_score', 'perspective_forced_en_pred_hate', 'perspective_forced_en_status', 'perspective_native_unsupported']
mBERT predictions present: 428 rows
Forced-EN predictions present: 789 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Classifier 2 — AfriHate (AfroXLMR)

Uses `Davlan/afro-xlmr-base-hate-v1` or the equivalent AfriHate release on
Hugging Face (Muhammad et al., 2025). Output labels are mapped to our binary
framing (predicted Hate vs. Not-Hate).


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

AFRIHATE_MODEL = 'Davlan/afro-xlmr-base-hate-v1'  # change if you prefer a different AfriHate checkpoint

tokenizer = AutoTokenizer.from_pretrained(AFRIHATE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(AFRIHATE_MODEL).to(DEVICE).eval()
id2label = model.config.id2label
print('AfriHate id2label:', id2label)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: Davlan/afro-xlmr-base-hate-v1 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
@torch.no_grad()
def afrihate_predict(texts, batch_size=32, max_length=256):
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc='AfriHate'):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors='pt').to(DEVICE)
        logits = model(**enc).logits
        preds = logits.argmax(dim=-1).cpu().numpy()
        for p in preds:
            results.append(id2label[int(p)])
    return results

afri_labels = afrihate_predict(df['OriginalText'].tolist())
df['afrihate_label'] = afri_labels

# Map AfriHate's output labels to our binary hate frame.
# AfriHate releases use labels like {abusive, hate, neutral, normal, offensive}.
# Any label containing 'hate' or 'abus' maps to Hate=True; everything else maps to False.
# Adjust this rule if your chosen checkpoint uses different label names.
def to_binary(lbl):
    s = str(lbl).lower()
    return any(kw in s for kw in ['hate', 'abus'])

df['afrihate_pred_hate'] = df['afrihate_label'].apply(to_binary)
df[['PostID', 'Language', 'afrihate_label', 'afrihate_pred_hate']].head()


## 5. Classifier 3 — Amharic mBERT

`amengemeda/amharic-hate-speech-detection-mBERT`. Applied to Amharic rows only.


In [ ]:
MBERT_MODEL = 'amengemeda/amharic-hate-speech-detection-mBERT'

tok_mb = AutoTokenizer.from_pretrained(MBERT_MODEL)
mdl_mb = AutoModelForSequenceClassification.from_pretrained(MBERT_MODEL).to(DEVICE).eval()
print('mBERT id2label:', mdl_mb.config.id2label)

@torch.no_grad()
def mbert_predict(texts, batch_size=32, max_length=256):
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Amharic mBERT'):
        batch = texts[i:i + batch_size]
        enc = tok_mb(batch, padding=True, truncation=True,
                     max_length=max_length, return_tensors='pt').to(DEVICE)
        logits = mdl_mb(**enc).logits
        preds = logits.argmax(dim=-1).cpu().numpy()
        for p in preds:
            results.append(int(p))
    return results

amh_mask = df['Language'] == 'Amharic'
amh_texts = df.loc[amh_mask, 'OriginalText'].tolist()
amh_preds = mbert_predict(amh_texts)

# The amengemeda checkpoint uses label id 1 for hate-speech; verify above.
df['mbert_pred_hate'] = None
df.loc[amh_mask, 'mbert_pred_hate'] = [bool(p == 1) for p in amh_preds]
df.loc[amh_mask, ['PostID', 'mbert_pred_hate']].head()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: amengemeda/amharic-hate-speech-detection-mBERT
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


mBERT id2label: {0: 'LABEL_0', 1: 'LABEL_1'}


Amharic mBERT:   0%|          | 0/14 [00:00<?, ?it/s]

,PostID,mbert_pred_hate
0,POST_0001,True
2,POST_0003,False
5,POST_0006,True
7,POST_0008,True
9,POST_0010,True


## 6. Save predictions

In [ ]:
OUT = 'predictions.xlsx'
cols = ['PostID', 'perspective_score', 'perspective_pred_hate', 'mbert_pred_hate']
df[cols].to_excel(OUT, index=False)
print(f'Saved: {OUT}, {len(df)} rows')
from google.colab import files
files.download(OUT)

Saved: predictions.xlsx, 789 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
OUT = 'predictions.xlsx'
cols = ['PostID', 'perspective_score', 'perspective_pred_hate',
        'afrihate_label', 'afrihate_pred_hate', 'mbert_pred_hate']
df[cols].to_excel(OUT, index=False)
print('Saved:', OUT)
files.download(OUT)


NameError: name 'df' is not defined

## 7. Next step — Stage 2 evaluation

Run `evaluate.py` locally (or in a fresh Colab notebook) against the cleaned
dataset and `predictions.xlsx`:

```
python evaluate.py \
    --dataset ethiopia_moderation_dataset_1000_cleaned.xlsx \
    --predictions predictions.xlsx \
    --out-xlsx evaluation_results.xlsx \
    --out-pdf figures.pdf
```

The script produces metrics, confusion matrices, error examples, and figures
for the paper.
